# BÁO CÁO THỰC HÀNH TUẦN 3 <br>
# Môn Học: Khai Thác Dữ liệu
``Họ và tên: Phạm Ngọc Hào`` <br>
``MSSV: 23110146``

Trong phần bài tập này, ta sẽ dùng ba hàm **Part1()**, **Part2()**, và **Part3()** để thực hiện các phần bài tập đã giao.

# 1. Khoảng cách thay đổi (Edit distance)

## Phát biểu bài toán

Cho hai chuỗi:
$$
 X = \{x_1, x_2, \ldots, x_n \} \textrm{ và } Y = \{y_1, y_2, \ldots, y_m \}
$$
Khoảng cách thay đổi là sự thay đổi được thực thi trong chuỗi $X$ thành chuỗi $Y$. Tìm khoảng cách thay đổi nhỏ nhất.

Ta gọi $f(i, j)$ là chi phí thấp nhất khi chuyển đổi $x_i$ thành $y_j$.

$$
f(i, j) =
\min \begin{cases}
f(i-1, j) + 1 \\
f(i, j-1) + 1 \\
f(i-1, j-1) + 
\begin{cases}
0, & x_i = y_j \\
2, & x_i \ne y_j
\end{cases}
\end{cases}
$$

Trong đó:
- $f(i, j)$ là chi phí nhỏ nhất để biến đổi chuỗi con $x_1 \ldots x_i$ thành $y_1 \ldots y_j$.
- $f(i-1, j) + 1$: xóa ký tự $x_i$ khỏi chuỗi $X$.
- $f(i, j-1) + 1$: chèn ký tự $y_j$ vào chuỗi $X$.
- $f(i-1, j-1)$:
  - Nếu $x_i = y_j$ thì không cần thao tác (chi phí $0$).
  - Nếu $x_i \ne y_j$ thì thay thế $x_i$ bằng $y_j$ (chi phí $2$).

Giá trị $f(i, j)$ được chọn là nhỏ nhất trong ba trường hợp trên.

Bài toán con cơ sở:
$$
f(0, j) = j, \quad f(i, 0) = i
$$


## Hàm tính khoảng cách thay đổi

In [1]:
def edit_distance(source_string, target_string):
    m = len(source_string)
    n = len(target_string)

    f = [[0] * (n + 1) for i in range(m + 1)]

    for i in range(1, m + 1):
        f[i][0] = i

    for j in range(1, n + 1):
        f[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if source_string[i-1] == target_string[j-1]:
                a = 0
            else:
                a = 2
            f[i][j] = min(f[i-1][j] + 1, f[i][j-1] + 1, f[i-1][j-1] + a)

    
    operations_performed = []
    i = len(source_string)
    j = len(target_string)

    while (i != 0 and j != 0):
        if source_string[i-1] == target_string[j-1]:
            i -=1
            j -=1

        else:
            if f[i][j] == f[i-1][j-1] + 2:
                operations_performed.append(
                    ("Replacement", source_string[i - 1], target_string[j - 1])
                )
                i -=1 
                j -=1

            elif f[i][j] == f[i-1][j] + 1:
                operations_performed.append(("Delete", source_string[i-1]))
                i -= 1
            
            elif f[i][j] == f[i][j-1] + 1:
                operations_performed.append(("Insert", target_string[j-1]))
                j -= 1

    while(j != 0):
        operations_performed.append(("Insert", target_string[j-1]))
        j -= 1

    while(i != 0):
        operations_performed.append(("Delete", source_string[i-1]))
        i -= 1

    operations_performed.reverse()
    return [f[m][n], operations_performed]

Hàm **edit_distance()** nhận hai chuỗi nguồn và chuỗi đích. Trước hết, ta khỏi tạo một bảng phương án kích thước $(m+1) \times (n+1)$ chứa toàn giá trị $0$, với $m$ là độ dài của chuỗi nguồn và $n$ là độ dài chuỗi đích. Tiếp theo đối với các toán con cơ sở, ta dùng vòng lặp **for** để điền các giá trị cơ sở vào bảng phương án như sau:
```python
for i in range(1, m + 1):
        f[i][0] = i

for j in range(1, n + 1):
    f[0][j] = j
```

Đến với phần công thức quy hoạch động, ta tiến hành điền vào bản phương án như theo công thức :
$$
f(i, j) =
\min \begin{cases}
f(i-1, j) + 1 \\
f(i, j-1) + 1 \\
f(i-1, j-1) + 
\begin{cases}
0, & x_i = y_j \\
2, & x_i \ne y_j
\end{cases}
\end{cases}
$$

```python
for i in range(1, m + 1):
    for j in range(1, n + 1):
        if source_string[i-1] == target_string[j-1]:
            a = 0
        else:
            a = 2
        f[i][j] = min(f[i-1][j] + 1, f[i][j-1] + 1, f[i-1][j-1] + a)
```

Ta duyệt từng phần tử của $m$ hàng và $n$ cột, nếu hai phần tử $i -1$ của chuổi nguồn bằng phần tử $j-1$ của chuỗi đích, ta gán $a =0$. Ngược lại, ta dùng công thức quy hoạch động để lưu kết quả cho bản phương án.
Đáp án của bài toán này là $f[m][n]$.

Sau khi xây dựng bảng quy hoạch động $f(i, j)$, ta tiến hành truy vết ngược từ $f(m, n)$ về $f(0, 0)$ để xác định các phép biến đổi đã thực hiện.

Bắt đầu từ $i = m$, $j = n$, lần ngược về $f(0,0)$:

- Nếu $x_i = y_j$: không có thao tác, di chuyển $(i-1, j-1)$
- Nếu $x_i \ne y_j$:
  - Nếu $f(i,j) = f(i-1,j-1) + 2$: **Replace**, di chuyển $(i-1, j-1)$
  - Nếu $f(i,j) = f(i-1,j) + 1$: **Delete**, di chuyển $(i-1, j)$
  - Nếu $f(i,j) = f(i,j-1) + 1$: **Insert**, di chuyển $(i, j-1)$

Sau khi một trong hai chỉ số về $0$:
- Nếu $j > 0$: thực hiện **Insert** các ký tự còn lại của $Y$
- Nếu $i > 0$: thực hiện **Delete** các ký tự còn lại của $X$

Cuối cùng, đảo ngược danh sách thao tác để thu được thứ tự biến đổi đúng.

In [2]:
import unittest
class TestEditDistance(unittest.TestCase):
    
    def test_identical_strings(self):
        dist, ops = edit_distance("hello", "hello")
        self.assertEqual(dist, 0)
        self.assertEqual(len(ops), 0)

    def test_replacement(self):
        dist, ops = edit_distance("a", "b")
        self.assertEqual(dist, 2)
        self.assertEqual(ops[0][0], "Replacement")

    def test_insertion_deletion(self):
        dist, ops = edit_distance("abc", "abcd")
        self.assertEqual(dist, 1)
        self.assertEqual(ops[0][0], "Insert")

    def test_complex_case(self):
        dist, ops = edit_distance("kitten", "sitting")
        self.assertEqual(dist, 5)

Phần kiểm thử (Unit Test)

Để đảm bảo tính đúng đắn của thuật toán, chương trình sử dụng thư viện `unittest` để xây dựng các trường hợp kiểm thử.

Các test case bao gồm:

- Trường hợp hai chuỗi giống nhau:
  - Ví dụ: "hello" → "hello"
  - Kết quả mong đợi: khoảng cách bằng 0, không có thao tác nào

- Trường hợp thay thế:
  - Ví dụ: "a" → "b"
  - Kết quả mong đợi: chi phí bằng 2, thực hiện phép Replacement

- Trường hợp chèn:
  - Ví dụ: "abc" → "abcd"
  - Kết quả mong đợi: chi phí bằng 1, thực hiện phép Insert

- Trường hợp phức tạp:
  - Ví dụ: "kitten" → "sitting"
  - Kết quả mong đợi: khoảng cách chỉnh sửa bằng 5

Các kiểm thử giúp xác nhận rằng:
- Công thức quy hoạch động được cài đặt chính xác
- Quá trình truy vết thao tác hoạt động đúng
- Thuật toán xử lý được cả trường hợp đơn giản và phức tạp


## Hàm xử lí yêu cầu 1:

In [3]:
def Part1():
    print("# Part 1:")
    source_string = "INTENTION"
    target_string = "EXECUTION"

    print(f"Source string: {source_string}")
    print(f"Target String: {target_string}")

    dist, operations_performed = edit_distance(source_string, target_string)

    insertions, deletions, replacements = 0, 0, 0
    for i in operations_performed:
        if i[0] == "Insert":
            insertions += 1
        elif i[0] == "Delete":
            deletions += 1
        else:
            replacements += 1
    print("Results")
    print(f"Minimum edit distance: {dist}")
    print(f"Number of insertions: {insertions}")
    print(f"Number of deletions: {deletions}")
    print(f"Number of replacements: {replacements}")
    print(f"Total number of operations: {insertions + deletions + replacements}")

    print("Actual operations")
    for i in range(len(operations_performed)):
        if operations_performed[i][0] == "Insert":
            print(
                f"{i+1}) {operations_performed[i][0]} : {operations_performed[i][1]}"
            )
        elif operations_performed[i][0] == "Delete":
            print(
                f"{i+1}) {operations_performed[i][0]} : {operations_performed[i][1]}"
            )
        else:
            print(
                f"{i+1}) {operations_performed[i][0]} : {operations_performed[i][1]} "
                f"by {operations_performed[i][2]}"
            )

    print()

Chương trình yêu cầu người dùng nhập vào:
- Chuỗi nguồn $X$
- Chuỗi đích $Y$

Sau đó, sử dụng hàm `edit_distance` để:
- Tính khoảng cách chỉnh sửa nhỏ nhất giữa hai chuỗi
- Truy vết và lưu lại danh sách các phép biến đổi đã thực hiện

``Xử lí kết quả``
Danh sách các thao tác được duyệt để thống kê:
- Số phép **Insert**
- Số phép **Delete**
- Số phép **Replacement**

Tổng số thao tác được tính bằng tổng của ba loại trên

``Hiển thị kết quả``
Chương trình in ra:
- Khoảng cách chỉnh sửa nhỏ nhất
- Số lượng từng loại thao tác
- Tổng số thao tác
- Danh sách chi tiết các thao tác theo thứ tự thực hiện

``Định dạng thao tác``
- Insert: `Insert : ký_tự`
- Delete: `Delete : ký_tự`
- Replacement: `Replacement : ký_tự_cũ by ký_tự_mới`

Các thao tác được đánh số thứ tự để dễ theo dõi.

# 2. Khoảng cách dãy con chung dài nhất (Longest Common Subsequence - LCS)

## Phát biểu bài toán

Cho hai chuỗi $X = x_1 x_2 \ldots x_m$ và $Y = y_1 y_2 \ldots y_n$.  
Tìm dãy con chung dài nhất (Longest Common Subsequence) của hai chuỗi.

Dãy con chung là dãy có thể thu được từ hai chuỗi ban đầu bằng cách xóa một số ký tự (không cần liên tiếp) nhưng vẫn giữ nguyên thứ tự.


Gọi $f(i, j)$ là độ dài dãy con chung dài nhất của:
- $x_1 \ldots x_i$
- $y_1 \ldots y_j$

Công thức quy hoạch động:

$$
f(i, j) =
\begin{cases}
f(i-1, j-1) + 1, & \text{nếu } x_i = y_j \\
\max(f(i-1, j), f(i, j-1)), & \text{nếu } x_i \ne y_j
\end{cases}
$$

Bài toán con cơ sở:
$$
f(i, 0) = 0, \quad f(0, j) = 0
$$

## Hàm tính khoảng cách dãy con chung dài nhất

In [4]:
def lcs(string1, string2):
    m = len(string1)
    n = len(string2)

    f = [[0] * (n+1) for i in range(m+1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if string1[i-1] == string2[j-1]:
                f[i][j] = f[i-1][j-1] + 1
            else:
                f[i][j] = max(f[i-1][j], f[i][j-1]) 

    operations_performed = []

    i = len(string1)
    j = len(string2)

    while(i != 0 and j != 0):
        if string1[i-1] == string2[j-1]:
            operations_performed.append(string1[i-1])
            i -= 1
            j -= 1
        
        else:
            if f[i-1][j] >= f[i][j-1]:
               i -=1
            else: j -=1

    operations_performed.reverse()
    return [f[m][n], operations_performed]

- Khởi tạo bảng $f$ kích thước $(m+1) \times (n+1)$, ban đầu bằng 0
- Duyệt qua từng cặp $(i, j)$:
  - Nếu hai ký tự trùng nhau → cộng thêm 1 từ $f(i-1, j-1)$
  - Nếu khác nhau → lấy giá trị lớn nhất từ:
    - Bỏ ký tự của chuỗi 1: $f(i-1, j)$
    - Bỏ ký tự của chuỗi 2: $f(i, j-1)$

Truy vết kết quả:

Bắt đầu từ $(m, n)$ để tìm lại dãy con chung:

- Nếu $x_i = y_j$:
  - Thêm ký tự đó vào kết quả
  - Di chuyển về $(i-1, j-1)$

- Nếu khác nhau:
  - Nếu $f(i-1, j) \ge f(i, j-1)$ → di chuyển lên $(i-1, j)$
  - Ngược lại → di chuyển sang trái $(i, j-1)$

Quá trình kết thúc khi $i = 0$ hoặc $j = 0$.

Cuối cùng đảo ngược kết quả để thu được đúng thứ tự.

Kết quả trả về:
- Độ dài LCS: $f(m, n)$
- Dãy con chung dài nhất (dưới dạng danh sách ký tự)

In [5]:
class TestLCS(unittest.TestCase):

    def test_common_subsequence(self):
        length, seq = lcs("ABCBDAB", "BDCABA")
        self.assertEqual(length, 4)
        self.assertEqual(len(seq), 4)

    def test_no_common(self):
        length, seq = lcs("abc", "def")
        self.assertEqual(length, 0)
        self.assertEqual(seq, [])

    def test_with_empty_string(self):
        length, seq = lcs("", "python")
        self.assertEqual(length, 0)
        self.assertEqual(seq, [])

Để kiểm tra tính đúng đắn của thuật toán LCS, chương trình sử dụng thư viện `unittest` với các trường hợp kiểm thử tiêu biểu:

- Trường hợp có dãy con chung:
  - Ví dụ: "ABCBDAB" và "BDCABA"
  - Kết quả mong đợi: độ dài LCS = 4
  - Kiểm tra cả độ dài và số phần tử của dãy kết quả

- Trường hợp không có ký tự chung:
  - Ví dụ: "abc" và "def"
  - Kết quả mong đợi: độ dài = 0, dãy kết quả rỗng

- Trường hợp có chuỗi rỗng:
  - Ví dụ: "" và "python"
  - Kết quả mong đợi: độ dài = 0, dãy kết quả rỗng

Các kiểm thử này đảm bảo rằng:
- Thuật toán hoạt động đúng với cả trường hợp thông thường và biên
- Xử lý chính xác khi không tồn tại dãy con chung
- Không xảy ra lỗi khi một trong hai chuỗi rỗng

## Hàm xử lý yêu cầu 2

In [6]:
def Part2():
    print("# Part 2:")
    string1 = "ACADB"
    string2 = "CBDA"

    print(f"The first string: {string1}")
    print(f"The second string: {string2}")

    lcs_magnitude, operations_performed = lcs(string1, string2)
    print(f"The magnitude of longest common subsequence: {lcs_magnitude}")
    print(f"The possible results: {operations_performed}")
    print()

# 3. Khoảng cách biến đổi thời gian động (Dynamic Time Warping- DTW)

## Phát biểu bài toán

Cho hai chuỗi số:
$$
X = (x_1, x_2, \ldots, x_m), \quad Y = (y_1, y_2, \ldots, y_n)
$$

Mục tiêu là tìm cách "căn chỉnh" hai chuỗi sao cho tổng chi phí là nhỏ nhất, cho phép co giãn trục thời gian (tức là một phần tử có thể ghép với nhiều phần tử bên kia).

Khoảng cách giữa hai phần tử được định nghĩa là:
$$
d(x_i, y_j) = |x_i - y_j|
$$



Gọi $f(i, j)$ là chi phí nhỏ nhất để căn chỉnh:
- $x_1 \ldots x_i$
- $y_1 \ldots y_j$

Công thức quy hoạch động:

$$
f(i, j) = \min \begin{cases}
f(i-1, j-1) \\
f(i-1, j) \\
f(i, j-1)
\end{cases} + |x_i - y_j|
$$

Bài toán con cơ sở:
$$
f(i, 0) = \infty, \quad f(0, j) = \infty
$$

## Hàm tính khoảng cách biến đổi thời gian động

In [7]:
def dtw(series1, series2):
    INF = float("inf")
    m = len(series1)
    n = len(series2)

    f = [[0]* (n + 1) for i in range(m + 1)]

    for i in range(1, m + 1):
        f[i][0] = INF

    for j in range(1, n + 1):
        f[0][j] = INF

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            d = float(abs(series1[i-1] - series2[j-1]))
            f[i][j] = min(f[i-1][j-1], f[i-1][j], f[i][j-1]) + d

    operations_performed = []
    i = len(series1)
    j = len(series2)

    while(i !=0 and j !=0):
        operations_performed.append(f[i][j])


        if f[i-1][j-1] <= f[i-1][j] and f[i-1][j-1] <= f[i][j-1]:
            i -=1
            j -=1

        elif f[i-1][j] <= f[i][j-1]:
            i -=1
        else:
            j -=1

    while i > 0:
        operations_performed.append(f[i][0])
        i -= 1
    
    while j > 0:
        operations_performed.append(f[0][j])
        j -= 1

    operations_performed.reverse()
    return [f[m][n], operations_performed]

- Khởi tạo bảng $f$ kích thước $(m+1) \times (n+1)$
- Gán các giá trị ở hàng 0 và cột 0 bằng vô cùng để tránh đường đi không hợp lệ
- Với mỗi cặp $(i, j)$:
  - Tính khoảng cách $d = |x_i - y_j|$
  - Chọn giá trị nhỏ nhất từ 3 hướng:
    - $(i-1, j-1)$: đi chéo (match)
    - $(i-1, j)$: đi lên (lặp phần tử của chuỗi 2)
    - $(i, j-1)$: đi trái (lặp phần tử của chuỗi 1)
  - Cộng thêm $d$ để cập nhật $f(i, j)$


Truy vết đường đi (Warping Path):

- Bắt đầu từ $(m, n)$
- Tại mỗi bước:
  - Thêm giá trị $f(i, j)$ vào danh sách kết quả
  - Di chuyển theo hướng có giá trị nhỏ nhất trong:
    - $(i-1, j-1)$
    - $(i-1, j)$
    - $(i, j-1)$

- Nếu còn dư phần tử:
  - Đi tiếp về $(0, 0)$ theo hàng hoặc cột

- Cuối cùng đảo ngược danh sách để có thứ tự đúng


Kết quả trả về:
- Tổng chi phí DTW: $f(m, n)$
- Đường đi căn chỉnh (dạng danh sách chi phí tích lũy)

In [8]:
class TestDTW(unittest.TestCase):

    def test_exact_match(self):
        s1 = [1.0, 2.0, 3.0]
        s2 = [1.0, 2.0, 3.0]
        dist, path = dtw(s1, s2)
        self.assertEqual(dist, 0.0)

    def test_warping_capability(self):
        s1 = [1, 2, 3]
        s2 = [1, 1, 2, 2, 3, 3]
        dist, path = dtw(s1, s2)
        self.assertEqual(dist, 0.0)

    def test_different_lengths(self):
        s1 = [1, 5, 8]
        s2 = [1, 9]
        dist, path = dtw(s1, s2)
        self.assertIsInstance(dist, float)
        self.assertTrue(len(path) > 0)

Để đảm bảo tính đúng đắn của thuật toán DTW, chương trình sử dụng thư viện `unittest` với các trường hợp kiểm thử tiêu biểu:

- Trường hợp hai chuỗi giống nhau:
  - Ví dụ: [1.0, 2.0, 3.0] và [1.0, 2.0, 3.0]
  - Kết quả mong đợi: khoảng cách DTW = 0.0

- Trường hợp có khả năng co giãn (warping):
  - Ví dụ: [1, 2, 3] và [1, 1, 2, 2, 3, 3]
  - Kết quả mong đợi: khoảng cách DTW = 0.0
  - Điều này chứng minh DTW cho phép một phần tử của chuỗi này khớp với nhiều phần tử của chuỗi kia

- Trường hợp độ dài khác nhau:
  - Ví dụ: [1, 5, 8] và [1, 9]
  - Kiểm tra:
    - Kết quả trả về là số thực (float)
    - Đường đi căn chỉnh (path) không rỗng

Các kiểm thử này giúp xác nhận rằng:
- Thuật toán xử lý đúng khi hai chuỗi trùng khớp hoàn toàn
- Cơ chế warping hoạt động chính xác
- Thuật toán vẫn hoạt động với chuỗi có độ dài khác nhau

## Hàm xử lí yêu cầu 3

In [9]:
def Part3():
    print("# Part 3:")
    line1 = "1 7 4 8 2 9 6 5 2 0"
    series1 = [float(x) for x in line1.split()]
    line2 = "1 2 8 5 5 1 9 4 6 5"
    series2 =[float(x) for x in line2.split()]

    print(f"The first series: {series1}")
    print(f"The second series: {series2}")

    min_dist, operations_performed = dtw(series1, series2)
    print(f"Min distance time warping: {min_dist}")
    operations_performed = list(map(int, operations_performed))
    print(f"The string path warping is: {operations_performed}")
    print()

# 4. Kiểm thử và thực thi chương trình

## Kiểm thử chương trình

In [10]:
unittest.main(argv = [''], verbosity = 2, exit = False)

test_different_lengths (__main__.TestDTW) ... ok
test_exact_match (__main__.TestDTW) ... ok
test_warping_capability (__main__.TestDTW) ... ok
test_complex_case (__main__.TestEditDistance) ... ok
test_identical_strings (__main__.TestEditDistance) ... ok
test_insertion_deletion (__main__.TestEditDistance) ... ok
test_replacement (__main__.TestEditDistance) ... ok
test_common_subsequence (__main__.TestLCS) ... ok
test_no_common (__main__.TestLCS) ... ok
test_with_empty_string (__main__.TestLCS) ... ok

----------------------------------------------------------------------
Ran 10 tests in 0.020s

OK


Chương trình đã được kiểm thử bằng thư viện `unittest` với tổng cộng 10 test case, bao gồm:
- 4 test cho bài toán Edit Distance
- 3 test cho bài toán LCS
- 3 test cho bài toán DTW

Kết quả thực thi:

- Tất cả các test đều chạy thành công (OK)
- Không phát hiện lỗi trong quá trình kiểm thử
- Thời gian thực thi rất nhỏ (0.019s), cho thấy thuật toán hoạt động hiệu quả với dữ liệu kiểm thử

Điều này chứng tỏ:
- Các thuật toán đã được cài đặt chính xác
- Các trường hợp cơ bản và biên đều được xử lý đúng
- Chương trình có độ tin cậy cao

Kết luận:
Hệ thống các thuật toán (Edit Distance, LCS, DTW) hoạt động đúng theo lý thuyết và đáp ứng yêu cầu đề bài.

## Thực thi chương trình

In [11]:
def main():
    # unittest.main(argv = [''], verbosity = 2, exit = False)
    Part1()
    Part2()
    Part3()

In [12]:
if __name__ == "__main__":
    main()

# Part 1:
Source string: INTENTION
Target String: EXECUTION
Results
Minimum edit distance: 8
Number of insertions: 1
Number of deletions: 1
Number of replacements: 3
Total number of operations: 5
Actual operations
1) Delete : I
2) Replacement : N by E
3) Replacement : T by X
4) Insert : C
5) Replacement : N by U

# Part 2:
The first string: ACADB
The second string: CBDA
The magnitude of longest common subsequence: 2
The possible results: ['C', 'A']

# Part 3:
The first series: [1.0, 7.0, 4.0, 8.0, 2.0, 9.0, 6.0, 5.0, 2.0, 0.0]
The second series: [1.0, 2.0, 8.0, 5.0, 5.0, 1.0, 9.0, 4.0, 6.0, 5.0]
Min distance time warping: 17.0
The string path warping is: [0, 1, 2, 3, 6, 7, 7, 9, 9, 9, 12, 17]

